In [3]:
import random

NUCLEOTIDES = ['A', 'C', 'G', 'T']

def complement(seq: str) -> str:
    return seq.translate(str.maketrans("ACGT", "TGCA"))

def reverse_complement(seq: str) -> str:
    return complement(seq)[::-1]

def random_dna(length: int) -> str:
    return "".join(random.choice(NUCLEOTIDES) for _ in range(length))

def create_transposon(tir: str, core_len: int) -> str:
    core = random_dna(core_len)
    return tir + core + reverse_complement(tir)

def insert_transposon(host_seq: str, pos: int, te_seq: str, tsd_len: int = 4):
    target = host_seq[pos : pos + tsd_len]
    new_seq = host_seq[:pos] + target + te_seq + target + host_seq[pos + tsd_len:]
    start_pos = pos
    end_pos = pos + len(target) + len(te_seq) + len(target)
    return new_seq, start_pos, end_pos

def detect_transposons(seq: str, tir_len: int = 5, tsd_len: int = 4, min_len: int = 25, max_len: int = 150):
    detected = []
    n = len(seq)
    for i in range(n - min_len):
        tsd_left = seq[i : i + tsd_len]
        tir_left = seq[i + tsd_len : i + tsd_len + tir_len]
        expected_tir_right = reverse_complement(tir_left)
        
        for j in range(i + min_len, min(i + max_len, n - tsd_len + 1)):
            tsd_right = seq[j : j + tsd_len]
            tir_right = seq[j - tir_len : j]
            
            if tsd_left == tsd_right and tir_right == expected_tir_right:
                detected.append({
                    "start": i,
                    "end": j + tsd_len,
                    "length": (j + tsd_len) - i,
                    "tsd": tsd_left,
                    "tir": tir_left
                })
    return detected

random.seed(42)

dna = random_dna(180)

te1 = create_transposon(tir="ATGCA", core_len=25)
te2 = create_transposon(tir="ATGCA", core_len=20)
te3 = create_transposon(tir="CGGTA", core_len=18)
te4 = create_transposon(tir="TTACC", core_len=15)

dna, s1, e1 = insert_transposon(dna, pos=30, te_seq=te1, tsd_len=4)
dna, s2, e2 = insert_transposon(dna, pos=s1 + 15, te_seq=te2, tsd_len=4)
dna, s3, e3 = insert_transposon(dna, pos=s2 + 10, te_seq=te3, tsd_len=4)
dna, s4, e4 = insert_transposon(dna, pos=len(dna) - 45, te_seq=te4, tsd_len=4)

print(f"Total DNA length: {len(dna)}")
print(f"Actual TE1 original insertion start: {s1}")
print(f"Actual TE2 (nested inside TE1): start={s2}, end={e2}")
print(f"Actual TE3 (nested inside TE2): start={s3}, end={e3}")
print(f"Actual TE4 (independent): start={s4}, end={e4}")

results = detect_transposons(dna, tir_len=5, tsd_len=4)
print(f"\nDetected {len(results)} elements:")
for r in results:
    print(f" - Range: [{r['start']}, {r['end']}], Length: {r['length']}, TSD: {r['tsd']}, TIR: {r['tir']}")

Total DNA length: 314
Actual TE1 original insertion start: 30
Actual TE2 (nested inside TE1): start=45, end=83
Actual TE3 (nested inside TE2): start=55, end=91
Actual TE4 (independent): start=240, end=273

Detected 4 elements:
 - Range: [30, 139], Length: 109, TSD: TAGG, TIR: ATGCA
 - Range: [45, 115], Length: 70, TSD: ACCC, TIR: ATGCA
 - Range: [55, 91], Length: 36, TSD: TTCT, TIR: CGGTA
 - Range: [240, 273], Length: 33, TSD: CCTC, TIR: TTACC
